# Segmentasi Pelanggan Zahir ? Tahap 1
Notebook berkelanjutan untuk audit awal dan EDA. Tahap ini tidak melakukan cleaning faktur, RFM, GMM, ekspor, atau integrasi Supabase. Output identitas hanya berupa agregat; jangan menampilkan DataFrame mentah atau daftar nilai identitas.

## 1. Pengumpulan dan pemuatan data
Sumber: `data/raw/Data Zahir 20k Data.xlsx`, sheet `faktur_penjualan_zahir (2)`. Satu faktur dapat memiliki beberapa baris item; nomor berulang belum berarti transaksi duplikat. Workbook dibuka baca-saja dan DataFrame mentah dipertahankan. Jalankan dari root repo menggunakan environment `venv_segmentasi_zahir`.


In [ ]:
from pathlib import Path
from datetime import datetime, date
import re
import io
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
SOURCE = ROOT / 'data' / 'raw' / 'Data Zahir 20k Data.xlsx'
SHEET = 'faktur_penjualan_zahir (2)'
REQUIRED = ['no_faktur', 'tanggal', 'nama_klien', 'contact_person', 'no_telepon',
            'nama_produk', 'qty', 'unit', 'harga_satuan', 'diskon_pct',
            'subtotal_produk', 'grand_total_faktur']
if not SOURCE.is_file():
    raise FileNotFoundError(f'Workbook tidak ditemukan: {SOURCE}. Jalankan notebook dari root repo.')
workbook_raw = pd.ExcelFile(SOURCE, engine='openpyxl')
if SHEET not in workbook_raw.sheet_names:
    raise ValueError(f'Sheet wajib {SHEET!r} tidak ditemukan. Periksa workbook sumber.')
df_raw = pd.read_excel(workbook_raw, sheet_name=SHEET)
missing_columns = sorted(set(REQUIRED) - set(df_raw.columns))
if missing_columns:
    raise ValueError(f'Kolom wajib tidak tersedia: {missing_columns}')
raw_snapshot = df_raw.copy(deep=True)
d = df_raw.copy(deep=True)
print('Interpreter:', sys.executable)
print('Sumber relatif:', SOURCE.relative_to(ROOT), '| Sheet:', SHEET)
print('Shape:', df_raw.shape)


## 2. Pemeriksaan awal dan statistik deskriptif
Head dibatasi pada atribut item yang aman. Statistik numerik tidak mencakup identitas; untuk kolom lain ditampilkan tipe, kelengkapan, dan jumlah unik tanpa nilai paling sering. String kosong atau hanya spasi dihitung sebagai kosong pada diagnosis, terpisah dari missing bawaan pandas.

Duplikat identik dihitung di **seluruh kolom asli**: `duplicated(keep=False)` menghitung semua anggota kelompok identik, sedangkan `duplicated(keep='first')` menghitung baris tambahan setelah kemunculan pertama. Tidak ada baris yang dihapus, sebab item yang sama bisa sah.


In [ ]:
SAFE = ['nama_produk', 'qty', 'unit', 'harga_satuan', 'diskon_pct', 'subtotal_produk', 'grand_total_faktur']
display(df_raw[SAFE].head())
buf = io.StringIO()
df_raw.info(buf=buf)
print(buf.getvalue())
display(df_raw[['harga_satuan', 'subtotal_produk', 'grand_total_faktur']].describe())
def blank(s):
    return s.isna() | s.map(lambda x: isinstance(x, str) and not x.strip())
profile = pd.DataFrame({
    'tipe': df_raw.dtypes.astype(str),
    'unik_terisi_asli': df_raw.nunique(dropna=True),
    'missing_pandas': df_raw.isna().sum(),
    'missing_persen': df_raw.isna().mean().mul(100),
    'kosong_termasuk_spasi': df_raw.apply(lambda s: blank(s).sum())
})
display(profile)
IDENTITY = ['nama_klien', 'contact_person', 'no_telepon', 'no_faktur']
display(profile.loc[IDENTITY])
identical_members = int(df_raw.duplicated(keep=False).sum())
identical_extra = int(df_raw.duplicated(keep='first').sum())
display(pd.DataFrame({'ukuran': ['Anggota kelompok baris identik', 'Baris identik tambahan'],
                      'jumlah': [identical_members, identical_extra]}))


## 3. EDA dan diagnosis faktur
Nomor faktur dikelompokkan menurut nilai asli yang terisi; spasi pada nomor tidak dibersihkan. Nomor kosong dikeluarkan dari pengelompokan dan dihitung sendiri. Jumlah nomor unik bukan jumlah faktur valid.

Tanggal native Excel dipertahankan. Teks ISO dibaca tahun-bulan-hari; teks numerik hari/bulan/tahun dibaca dengan asumsi **day-first**. Teks yang hari dan bulannya sama-sama 1?12 ditandai ambigu meskipun berhasil diparsing. Angka nonkosong tidak otomatis dianggap serial Excel. Format lain yang tidak didukung ditandai gagal untuk ditinjau. Tidak ada nilai tanggal asli yang ditampilkan.


In [ ]:
invoice_blank = blank(d['no_faktur'])
g = d.loc[~invoice_blank].groupby('no_faktur', sort=False, dropna=False)
rows_per_invoice = g.size()
display(pd.DataFrame({'ukuran': ['Baris mentah', 'Baris tanpa nomor', 'Nomor faktur unik terisi', 'Nomor dengan >1 baris'],
                      'jumlah': [len(d), int(invoice_blank.sum()), len(rows_per_invoice), int(rows_per_invoice.gt(1).sum())]}))
row_distribution = rows_per_invoice.value_counts().sort_index().rename_axis('baris_per_nomor').to_frame('jumlah_nomor')
display(row_distribution)
display(rows_per_invoice.describe().to_frame('baris_per_nomor'))

def parse_diagnostic(v):
    if pd.isna(v) or (isinstance(v, str) and not v.strip()):
        return pd.NaT, 'kosong', False
    if isinstance(v, (datetime, date, pd.Timestamp)):
        return pd.Timestamp(v), 'native_datetime', False
    if isinstance(v, str):
        s = v.strip()
        if re.fullmatch(r'\d{4}-\d{1,2}-\d{1,2}(?:[ T].*)?', s):
            return pd.to_datetime(s, errors='coerce'), 'teks_ISO', False
        match = re.fullmatch(r'(\d{1,2})[/.-](\d{1,2})[/.-](\d{4})', s)
        if match:
            a, b, y = map(int, match.groups())
            parsed = pd.to_datetime(f'{y:04d}-{b:02d}-{a:02d}', errors='coerce')
            return parsed, 'teks_dayfirst', 1 <= a <= 12 and 1 <= b <= 12
    return pd.NaT, 'format_tidak_didukung', False
parsed = d['tanggal'].map(parse_diagnostic)
d['_tanggal'] = parsed.map(lambda x: x[0])
d['_format_tanggal'] = parsed.map(lambda x: x[1])
d['_ambigu'] = parsed.map(lambda x: x[2])
date_blank = blank(d['tanggal'])
date_failed = ~date_blank & d['_tanggal'].isna()
display(d['tanggal'].map(lambda x: type(x).__name__).value_counts().rename_axis('tipe_python').to_frame('baris'))
display(pd.crosstab(d['_format_tanggal'], d['_tanggal'].notna(), dropna=False).rename(columns={False: 'tidak_terparsing', True: 'terparsing'}))
# Bentuk teks mengungkap format tanpa menampilkan nilai tanggal asli.
text_dates = d.loc[d['tanggal'].map(lambda x: isinstance(x, str)), 'tanggal']
display(text_dates.map(lambda x: re.sub(r'[0-9]', 'D', x)).value_counts().rename_axis('pola_teks_angka_disamarkan').to_frame('baris'))
display(pd.DataFrame({'ukuran': ['Tanggal kosong', 'Tanggal terisi gagal parsing', 'Tanggal teks ambigu'],
                      'baris': [int(date_blank.sum()), int(date_failed.sum()), int(d['_ambigu'].sum())]}))


In [ ]:
# Konflik hanya membandingkan nilai terisi. Kosong dan gagal parsing dicatat terpisah.
conflicts = []
for label, source_col, value_col in [('klien', 'nama_klien', 'nama_klien'),
                                      ('tanggal', 'tanggal', '_tanggal'),
                                      ('grand_total', 'grand_total_faktur', 'grand_total_faktur')]:
    values = d[value_col].mask(blank(d[source_col]))
    temp = pd.DataFrame({'nomor': d['no_faktur'], 'nilai': values,
                         'kosong': blank(d[source_col]),
                         'gagal': date_failed if label == 'tanggal' else False}).loc[~invoice_blank]
    grouped = temp.groupby('nomor', sort=False)
    n = grouped['nilai'].nunique()
    has_blank = grouped['kosong'].any()
    has_filled = grouped['nilai'].count().gt(0)
    conflicts.append({'atribut': label, 'konflik_nilai_terisi': int(n.gt(1).sum()),
                      'nomor_dengan_kosong': int(has_blank.sum()),
                      'nomor_kosong_dan_terisi': int((has_blank & has_filled).sum()),
                      'nomor_dengan_gagal_parsing': int(grouped['gagal'].any().sum())})
conflict_table = pd.DataFrame(conflicts).set_index('atribut')
display(conflict_table)
# Pengulangan nilai total pada item: tidak menjumlahkan grand_total_faktur.
total_stats = g['grand_total_faktur'].agg(['count', 'nunique'])
repeated_total_numbers = int((total_stats['count'].gt(total_stats['nunique'])).sum())
same_total_multiline = int((rows_per_invoice.gt(1) & total_stats['nunique'].eq(1) & total_stats['count'].eq(rows_per_invoice)).sum())
display(pd.DataFrame({'ukuran': ['Nomor dengan total terisi berulang', 'Nomor multibaris dengan semua total sama dan terisi'],
                      'jumlah': [repeated_total_numbers, same_total_multiline]}))
subtotal = pd.to_numeric(d['subtotal_produk'], errors='coerce')
display(subtotal.describe(percentiles=[.01, .25, .5, .75, .95, .99]).to_frame('subtotal_tingkat_baris'))
display(pd.Series({'subtotal_kosong': int(blank(d['subtotal_produk']).sum()),
                   'subtotal_negatif': int(subtotal.lt(0).sum()), 'subtotal_nol': int(subtotal.eq(0).sum())}).to_frame('baris'))
product_blank = blank(d['nama_produk'])
products = d.loc[~product_blank, 'nama_produk'].astype(str)
product_norm = products.str.strip().str.replace(r'\s+', ' ', regex=True).str.casefold()
product_pairs = pd.DataFrame({'asli': products, 'normal': product_norm}).drop_duplicates()
variants = product_pairs.groupby('normal')['asli'].nunique()
variant_groups = int(variants.gt(1).sum())
variant_rows = int(product_norm.isin(variants[variants.gt(1)].index).sum())
display(pd.Series({'baris_produk_kosong': int(product_blank.sum()), 'nama_asli_unik': products.nunique(),
                   'nama_normalisasi_unik': product_norm.nunique(), 'kelompok_dengan_variasi': variant_groups,
                   'baris_dalam_kelompok_variasi': variant_rows}).to_frame('jumlah'))


Konflik dapat tumpang tindih: satu nomor bisa mempunyai beberapa klien, tanggal, sekaligus total. Tanggal dibandingkan setelah parsing diagnostik; kasus ambigu tetap memerlukan validasi sumber. Klien dibandingkan persis sesuai nilai asli. Pengulangan total bisa merupakan total faktur yang disalin ke setiap item; **jumlah grand_total_faktur mentah tidak dipakai sebagai penjualan**. Subtotal ditelaah pada tingkat baris, tanpa menetapkan aturan diskon atau rekonsiliasi. Normalisasi nama produk hanya mengukur indikasi variasi; produk belum digabungkan.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
profile['kosong_termasuk_spasi'].div(len(d)).mul(100).sort_values().plot.barh(ax=axes[0])
axes[0].set(title='Kelengkapan kolom', xlabel='Nilai kosong (%)')
row_distribution['jumlah_nomor'].plot.bar(ax=axes[1])
axes[1].set(title='Jumlah baris per nomor faktur', xlabel='Baris per nomor', ylabel='Jumlah nomor', yscale='log')
axes[1].tick_params(axis='x', labelsize=7)
axes[2].hist(np.arcsinh(subtotal.dropna()), bins=50)
axes[2].set(title='Distribusi subtotal tingkat baris', xlabel='arcsinh(subtotal), termasuk nol/negatif', ylabel='Jumlah baris')
plt.tight_layout()
display(fig)
plt.close(fig)


## 4. Ringkasan audit dan keputusan untuk Tahap 2
Tabel berikut dihitung dari workbook saat notebook dijalankan. Unit kasus dinyatakan eksplisit; jumlah antarbaris tabel tidak boleh dijumlahkan karena dapat tumpang tindih.


In [ ]:
audit_rows = []
def audit(kind, count, unit, impact, action):
    audit_rows.append({'pemeriksaan': kind, 'jumlah_kasus': int(count), 'unit': unit,
                       'dampak_RFM': impact, 'validasi_Tahap_2': action})
audit('Baris identik tambahan', identical_extra, 'baris', 'Potensi penggelembungan nilai/frekuensi', 'Periksa ID item dan bukti sumber sebelum menghapus')
audit('Anggota kelompok identik', identical_members, 'baris', 'Menunjukkan cakupan duplikasi', 'Bedakan item sah dari duplikasi ekstraksi')
audit('Nomor faktur kosong', invoice_blank.sum(), 'baris', 'Frekuensi tidak dapat ditautkan', 'Telusuri nomor atau ID transaksi sumber')
audit('Nomor multibaris', rows_per_invoice.gt(1).sum(), 'nomor', 'Jumlah baris bukan Frequency', 'Validasi grain item dan aturan agregasi')
for label, row in conflict_table.iterrows():
    audit(f'Konflik {label} antar-nilai terisi', row['konflik_nilai_terisi'], 'nomor', 'Identitas/grain atau nilai RFM tidak pasti', 'Periksa ID transaksi, cabang, seri, dan dokumen sumber')
    audit(f'{label}: nomor dengan nilai kosong', row['nomor_dengan_kosong'], 'nomor', 'Atribut RFM tidak lengkap', 'Validasi sumber; jangan isi otomatis')
audit('Tanggal kosong', date_blank.sum(), 'baris', 'Recency tidak tersedia', 'Konfirmasi tanggal transaksi sumber')
audit('Tanggal gagal parsing', date_failed.sum(), 'baris', 'Recency tidak dapat dihitung', 'Periksa format/serial Excel dan lokal sumber')
audit('Tanggal ambigu', d['_ambigu'].sum(), 'baris', 'Recency mungkin bergeser', 'Konfirmasi lokal day-first dengan pemilik data')
audit('Total berulang dalam nomor', repeated_total_numbers, 'nomor', 'Monetary membesar jika total dijumlahkan', 'Rekonsiliasi total faktur dan subtotal item')
audit('Subtotal negatif', subtotal.lt(0).sum(), 'baris', 'Retur/koreksi memengaruhi Monetary', 'Periksa status retur dan pembatalan')
audit('Nama produk kosong', product_blank.sum(), 'baris', 'Interpretasi komposisi belanja terbatas', 'Telusuri kode produk sumber')
audit('Kelompok variasi nama produk', variant_groups, 'kelompok normalisasi', 'Kategori belanja dapat terpecah', 'Validasi kode produk sebelum penggabungan')
audit_table = pd.DataFrame(audit_rows)
display(audit_table)
key_conflicts = int(conflict_table['konflik_nilai_terisi'].sum())
if key_conflicts:
    print('no_faktur belum layak ditetapkan sebagai kunci faktur tunggal: terdapat konflik antar-nilai terisi.')
else:
    print('Belum ada konflik antar-nilai terisi yang terdeteksi, tetapi keunikan bisnis no_faktur tetap perlu dikonfirmasi.')
display(pd.DataFrame({'metrik': ['Jumlah baris mentah', 'Jumlah nomor faktur unik terisi', 'Jumlah faktur valid'],
                      'hasil': [str(len(d)), str(len(rows_per_invoice)), 'Belum ditetapkan; menunggu aturan cleaning dan validasi kunci']}))
assert df_raw.equals(raw_snapshot), 'DataFrame mentah berubah selama diagnosis.'
verification_summary = {'baris_mentah': len(d), 'nomor_unik': len(rows_per_invoice),
                        'identik_tambahan': identical_extra, 'identik_anggota': identical_members,
                        'konflik': conflict_table.to_dict('index'), 'tanggal_kosong': int(date_blank.sum()),
                        'tanggal_gagal': int(date_failed.sum()), 'tanggal_ambigu': int(d['_ambigu'].sum()),
                        'nomor_total_berulang': repeated_total_numbers, 'variasi_produk': variant_groups}
print(json.dumps(verification_summary, ensure_ascii=False))


Untuk Tahap 2, minta ID faktur internal yang stabil, ID pelanggan, cabang/perusahaan, seri atau periode nomor, status transaksi, dan ID baris item. Kandidat kunci komposit seperti `(perusahaan/cabang, seri, no_faktur)` harus diuji terhadap dokumen sumber. Kombinasi nomor, klien, dan tanggal hanya kandidat investigasi, karena berisiko memisahkan satu faktur saat atribut salah atau kosong.

Tahan penggabungan, penghapusan, pemilihan satu grand total, serta penetapan jumlah faktur valid sampai grain data, konflik, tanggal ambigu, dan aturan retur disepakati. Frequency harus menghitung faktur yang telah divalidasi; Recency memakai tanggal tervalidasi; Monetary memakai nilai faktur yang sudah direkonsiliasi. Tahap selanjutnya ditambahkan ke notebook yang sama.
